# 03 — The built dataset

Notebook 02 built it. This one reads it back and reports what is there.

Nothing is written.

## Configuration

In [ ]:
from pathlib import Path

REPO_ROOT = Path.cwd().parent

# INPUT, must exist. Built by notebooks/02_build_dataset.ipynb.
DATASET = REPO_ROOT / "dataset" / "multi_subtype_80mm"
METADATA_CSV = DATASET / "metadata.csv"       # one row per image
IMAGES_DIR = DATASET / "images"               # <patient>/slice_NNN.png
CONFIG_JSON = DATASET / "config.json"         # the settings that produced it

# Fixed order. Every model, confusion matrix and per-class metric in the project
# indexes on it, so it is written out rather than read off the data, where it
# would follow whatever order pandas happened to produce.
CLASSES = ["HRposHER2neg", "TripleNeg", "HER2pos"]
COHORTS = ["duke", "spy1", "spy2"]

print(f"dataset  {DATASET}")
print(f"exists   {METADATA_CSV.is_file()}")

## Imports and data

In [ ]:
import json

import matplotlib.pyplot as plt
import pandas as pd
from PIL import Image

meta = pd.read_csv(METADATA_CSV)
cfg = json.loads(CONFIG_JSON.read_text())

# `meta` is one row per IMAGE. Anything counting patients has to drop duplicates
# on pid first, or every count comes out multiplied by the slices per patient.
patients = meta.drop_duplicates("pid")

print(f"images   {len(meta):,}")
print(f"patients {len(patients):,}")
print(f"built with: {cfg['n_slices']} slices per patient, "
      f"{cfg['crop_mm']:.0f} mm crop, {cfg['save_size']}x{cfg['save_size']} px, "
      f"{cfg['normalization']} normalisation")

## Composition

In [ ]:
by_cohort = pd.DataFrame({
    "patients": patients.groupby("cohort").size(),
    "images": meta.groupby("cohort").size(),
})
# Below the configured n_slices because a slice with too little tumour in it is
# dropped at build time.
by_cohort["slices_per_patient"] = (by_cohort.images / by_cohort.patients).round(2)
by_cohort.loc["TOTAL"] = [len(patients), len(meta),
                          round(len(meta) / len(patients), 2)]
print(by_cohort.to_string())

# Duke ships no voxel mask, so its region of interest is an expert-drawn box.
print()
print(patients.groupby(["cohort", "roi_source"]).size().to_string())

## Splits and the trivial baseline

In [ ]:
rows = []
for split in ("train", "val", "test"):
    p = patients[patients.split == split]
    counts = p.label_name.value_counts()
    rows.append({
        "split": split,
        "patients": len(p),
        "images": int((meta.split == split).sum()),
        **{c: int(counts.get(c, 0)) for c in CLASSES},
        # Accuracy of always predicting the majority class OF THIS SPLIT.
        # It differs per split, so it is computed here rather than quoted.
        "trivial_baseline": round(counts.max() / len(p), 4),
    })
print(pd.DataFrame(rows).to_string(index=False))

# Asserted, not printed. A patient in two splits leaves the metrics looking
# normal, so this notebook fails rather than reports if either breaks.
overlap = patients.pid[patients.pid.duplicated()].tolist()
assert not overlap, f"patient in two splits: {overlap[:5]}"
assert meta.groupby("pid").label.nunique().max() == 1, "a patient carries two labels"
print("\nno patient in two splits, no patient with two labels")

## The cohorts after preprocessing

In [ ]:
# Preprocessing made every image the same size and resolution. It did not make
# the cohorts the same, and a pooled result carries the difference with it.
share = (patients.groupby("cohort").label_name.value_counts(normalize=True)
         .unstack().reindex(columns=CLASSES).round(3) * 100)
print("class share per cohort, %")
print(share.to_string())
print(f"\nspread on {CLASSES[0]}: "
      f"{share[CLASSES[0]].max() - share[CLASSES[0]].min():.1f} percentage points")

vol = patients.groupby("cohort").tum_vol.median().round(1)
print("\nmedian tumour volume per cohort")
print(vol.to_string())
print(f"\nlargest / smallest: {vol.max() / vol.min():.1f}x")

## The same thing as a picture

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(13, 3.4))
colour = ["#0072B2", "#D55E00", "#009E73"]

by_cohort.drop("TOTAL").patients.plot.bar(ax=ax[0], color="#0072B2", rot=0)
ax[0].set_title("patients per cohort")
ax[0].set_xlabel("")

share.plot.bar(ax=ax[1], color=colour, rot=0, width=0.8)
ax[1].set_title("class share per cohort (%)")
ax[1].set_xlabel("")
ax[1].legend(fontsize=7, frameon=False)

# Log scale: Duke's median is about a fifth of the others, and a linear axis
# flattens the two larger cohorts into the same bar.
ax[2].bar(vol.index, vol.values, color="#0072B2")
ax[2].set_yscale("log")
ax[2].set_title("median tumour volume (log)")

for a in ax:
    a.spines[["top", "right"]].set_visible(False)
fig.tight_layout()
plt.show()

## Resolution

In [ ]:
# This is what the 80 mm crop buys. Source spacing varies several-fold, so a
# crop measured in pixels would cover a different amount of tissue per patient
# and tumour size would stop being comparable between them.
print(f"source xy_spacing  {meta.xy_spacing.min():.3f} to "
      f"{meta.xy_spacing.max():.3f} mm/px   "
      f"({meta.xy_spacing.max() / meta.xy_spacing.min():.1f}x)")
print(f"crop side in pixels {int(meta.crop_px.min())} to {int(meta.crop_px.max())}")
print(f"final mm per pixel  {meta.mm_per_px.min():.5f} to {meta.mm_per_px.max():.5f}"
      f"   ({meta.mm_per_px.max() / meta.mm_per_px.min():.4f}x)")
print(f"final image size    {int(meta.img_size.min())}x{int(meta.img_size.max())} px,"
      f" every image")

## What the network is handed

In [ ]:
# Real files from images/. The three colour channels are three acquisition
# time-points, so the colour of a voxel is how it took up and released the
# contrast agent.
fig, axes = plt.subplots(len(COHORTS), len(CLASSES), figsize=(7.5, 7.5))
for r, cohort in enumerate(COHORTS):
    for c, cls in enumerate(CLASSES):
        ax = axes[r][c]
        ax.axis("off")
        pick = meta[(meta.cohort == cohort) & (meta.label_name == cls)]
        if pick.empty:
            ax.set_title(f"{cohort} / {cls}\nnone", fontsize=7)
            continue
        row = pick.iloc[len(pick) // 2]
        # `filename` is relative to images/, not to the dataset root.
        ax.imshow(Image.open(IMAGES_DIR / row.filename))
        ax.set_title(f"{cohort} / {cls}", fontsize=8)
fig.suptitle("R = pre-contrast, G = early post, B = late post", fontsize=9)
fig.tight_layout()
plt.show()

## What this notebook established

2,063 patients and 16,378 images, every one 224x224 at a constant 0.357 mm per pixel.

The splits are clean, asserted above rather than reported.

Each split has its own trivial baseline. The test split is 0.5112.

The cohorts still differ after preprocessing, by about 26 percentage points on class
share and about five times on median tumour volume.

Next: [`04_train_centralized.ipynb`](04_train_centralized.ipynb).